<a href="https://colab.research.google.com/github/MAI-AIIU/advanced-programming/blob/main/assignments/first/src/First_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building an AI-Ready Dataset Using Web Scraping, External Datasets, NumPy, and Pandas.

In this assignment, students will collect data from the web using web scraping and combine it with another external
dataset. They will clean, transform, analyze, and prepare the data using Pandas and NumPy.

## Scraping `books.toscrape.com` website
Loading the required libraries, which I use `requests` to send request and pull data and using `BeautifulSoup` to scrape data from the HTML and CSS data I have recieved via request

#### Load libraries

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

#### Define base_url and headers

In [2]:
BASE_URL = "https://books.toscrape.com/"

# we use this header to avoid flagging our request as automated by the website
headers = {
    "User-Agent": "Mozilla/5.0"
}

#### Define mappers to map data from original form to another form
in below example we map ratings from text to numbers

In [3]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

#### Defining global variables

In [4]:
books = []
next_page = BASE_URL
page = 5 # I put this variable to only load first 5 pages.

In [ ]:
while next_page:
    print(f"Scraping {next_page}")

    response = requests.get(next_page, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.select("article.product_pod"):

        title = book.h3.a["title"]
        price = book.select_one("p.price_color").text.strip()

        availability = (
            book.select_one("p.instock.availability")
            .get_text(strip=True)
        )

        rating = rating_map[
            book.select_one("p.star-rating")["class"][1]
        ]

        # Product URL
        relative_url = book.h3.a["href"]
        product_url = urljoin(next_page, relative_url)

        # Visit product page to get category
        detail_response = requests.get(product_url, headers=headers)
        detail_response.raise_for_status()

        detail_soup = BeautifulSoup(detail_response.text, "html.parser")

        category = (
            detail_soup.select("ul.breadcrumb li a")[2]
            .get_text(strip=True)
        )

        books.append({
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Availability": availability,
            "Category": category,
            "Product Link": product_url
        })

    # Find next page
    next_button = soup.select_one("li.next a")

    if next_button and page > 1:
        page -= 1
        next_page = urljoin(next_page, next_button["href"])
    else:
        next_page = None

toscrape_books = pd.DataFrame(books)
print(f"Scraped {len(books)} books.")
print("Saved to books.csv.")

Scraping https://books.toscrape.com/
Scraping https://books.toscrape.com/catalogue/page-2.html


#### Export to CSV file

In [ ]:
toscrape_books.to_csv("toscrape_books.csv", index=False)

## Loading books csv file from Github

Note: As Kaggle required to authenticate before loading dataset, and this dataset was the matched one with the one I scraped from `https://books.toscrape.com/`, so I downloaded its CSV and uploaded to my own Github repo, and put the code below to pull to Colab Notebook.

In [ ]:
url = "https://raw.githubusercontent.com/MAI-AIIU/advanced-programming/main/assignments/first/files/github_books.csv"

github_books = pd.read_csv(url)

#### Rename column names to match `toscrape_books` with `github_books`

In [ ]:
github_books.columns, toscrape_books.columns # matching both columns

In [ ]:
# this will rename and change the order of columns to match
github_books = (
    github_books
    .rename(columns={
        "title": "Title",
        "price": "Price",
        "rating": "Rating",
        "stock": "Availability",
        "category": "Category",
        "book_url": "Product Link"
    })
    [["Title", "Price",  "Rating", "Availability", "Category", "Product Link"]]
)

In [ ]:
github_books.columns, toscrape_books.columns # matching both columns

#### Export to CSV file

In [ ]:
github_books.to_csv("github_books.csv", index=False)

## Pandas Data Cleaning and Preparation

#### Explain both datasets

In [ ]:
toscrape_books.head(1), github_books.head(1)

In [ ]:
# prints dataset columns

toscrape_books.columns, github_books.columns

In [ ]:
# prints datasets information such as columns, non-null counts, data type of each column, memory usage, range of data
toscrape_books.info(), github_books.info()

In [ ]:
# prints dataset shapes, the first number in tuple is number of rows, and the second is number of columns
toscrape_books.shape, github_books.shape

In [ ]:
# print top 3 rows of first dataset and 3 rows from end from second dataset
toscrape_books.head(3), github_books.tail(3)

In [ ]:
# describe prints some statistics from both datasets like count, mean, std, min, 25%, and also prints count, unique, top, ...
toscrape_books.describe(), github_books.describe()

In [ ]:
github_books.isna().sum(), toscrape_books.isna().sum()

#### Convert Data type
This task should be done later, but as handling missing values are having issue drawing histogram so we need to convert it first.

In [ ]:
github_books["Price"] = github_books["Price"].str.replace("Â£", "").astype(float)
toscrape_books["Price"] = toscrape_books["Price"].str.replace("Â£", "").astype(float)
github_books.head()

#### Handling missing values

As per above analysis, scrape books don't have any missing value but Github books have some, so  
1: filling missing values in Price and Ratings  
`we can fill missing values with mean, media, mode, regression, KNN or constant value`  
2: dropping records having missing values on Title, because Title is name of the book and it is unique for each record. so there is no benifitial way to fill it.

In [ ]:
import matplotlib.pyplot as plt
github_books['Price'].hist()
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
print(github_books["Price"].skew())

In [ ]:
print("Mean:", github_books["Price"].mean())
print("Median:", github_books["Price"].median())

Based on the skewness which is `-0.12`, and mean and median which is approximately close and the histogram shape, we can decide to use `mean imputation`

In [ ]:
github_books["Price"] = github_books["Price"].fillna(github_books["Price"].mean())

Now let's check if the price have still null or not.

In [ ]:
github_books.isna().sum()

Now let's fill category using `Mode imputation`, and that is the mlogical way to fill the text fields

In [ ]:
github_books["Category"] = github_books["Category"].fillna(
    github_books["Category"].mode().iloc[0]
)

In [ ]:
github_books.isna().sum()

Now let's drop the missing title records

In [ ]:
github_books = github_books.dropna(subset=["Title"])

In [ ]:
github_books.isna().sum()

#### Remove Duplicate Records

In [ ]:
github_books.duplicated().sum(), toscrape_books.duplicated().sum()

No duplicate record found

#### Clean Text Columns
This is already done for the Price column above, and there is no other field for that.

#### Convert Data Types

Now mapping `{1, 2, 3, 4, 5}` to `{one, two, three, four, five}` for the rate field.   
For the scraped books it is already done, only applying for the github books.

In [ ]:
github_books["Rating"] = github_books["Rating"].map(rating_map)

In [ ]:
github_books.head(5), toscrape_books.head(5)

#### Filtering

In [ ]:
not_lower_not_upper_rating = github_books[(github_books["Rating"] >= 2) & (github_books["Rating"] < 4)]
available_books_with_lower_rating = github_books[github_books["Rating"] <= 2 github_books["Availability"] == "In stock"]
available_books_with_price_less_than_the_average_price = github_books[(github_books["Price"] < github_books["Price"].mean())]
available_books_with_price_less_than_the_mode_price = github_books[(github_books["Price"] < github_books["Price"].mode())]

#### Sorting

In [ ]:
github_books.sort_values("Title", ascending=False)
github_books.sort_values(["Price", "Rating"], ascending=[False, True])

#### Grouping and Aggregation

In [ ]:
github_books.groupby("Category")["Price"].mean()
github_books.groupby("Rating")["Title"].count()
github_books.groupby("Availability")["Price"].agg(["mean", "min", "max"])

#### Feature Engineering

In [ ]:
github_books["title_hash"] = hash(github_books["Title"])
github_books["rate_price_value"] = github_books["Price"] * github_books["Rating"]
github_books["rating_stars"] = github_books["Rating"].apply(
    lambda x: "⭐" * x
)

price_mean = github_books["Price"].mean()

github_books["price_category"] = github_books["Price"].apply(
    lambda x: "Expensive" if x > price_mean else "Cheap"
)